# LAB Ensembles: Bagging, Boosting e Stacking no dataset Movies (Vega)

**Equipe:**
Guilherme Schnekenberg Teixeira (RA00336189), Luís Augusto Coelho de Souza (RA00331675)

Continuação do LAB de Decision Trees, disponível no repositório especificado no fim da célula. O problema é o mesmo: prever, antes do lançamento, se um filme será um **sucesso comercial** (`Worldwide_Gross > Production_Budget`). Reaproveitamos todas as decisões do EDA anterior (correção de século, filtro 1996+, casos completos, `release_season`, split temporal) e comparamos três ensembles com a árvore única escolhida antes:

- **Bagging:** `BaggingClassifier` com árvores de decisão
- **Boosting:** `HistGradientBoostingClassifier`
- **Stacking:** `StackingClassifier` combinando árvore, bagging, boosting e regressão logística

Métricas principais: **AUC** e **balanced accuracy**, porque 70,4% dos filmes de teste são sucessos e a acurácia sozinha engana.

**Repositório Decision Trees:** https://github.com/Luis-coelho30/movie-success-prediction.git

In [ ]:
import numpy as np
import pandas as pd
import itertools
from vega_datasets import data

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import BaggingClassifier, HistGradientBoostingClassifier, StackingClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, roc_auc_score,
                             f1_score, confusion_matrix)

df = data.movies().copy()

# 1. Datas: corrige o erro de século
d = pd.to_datetime(df["Release_Date"], format="%b %d %Y", errors="coerce")
year = d.dt.year
df["release_year_fixed"] = year.where(year <= 2011, year - 100)
df["release_month"] = d.dt.month
df["Release_Date_fixed"] = d.where(year <= 2011, d - pd.DateOffset(years=100))

# 2. Zeros viram nulos, remove linhas sem rótulo e cria o target
df[["Worldwide_Gross", "Production_Budget"]] = (
    df[["Worldwide_Gross", "Production_Budget"]].replace(0, np.nan))
df = df.dropna(subset=["Worldwide_Gross", "Production_Budget"])
df["success"] = (df["Worldwide_Gross"] > df["Production_Budget"]).astype(int)

# 3. Filtro de época
df = df[df["release_year_fixed"] >= 1996]

# 4. Agrupamento de categorias raras
rare_source = ["Traditional/Legend/Fairytale", "Based on Magazine Article", "Based on Musical/Opera",
               "Based on Short Film", "Spin-Off", "Based on Factual Book/Article",
               "Disney Ride", "Compilation", "Based on Toy", "Musical Group Movie"]
df["MPAA_Rating"] = df["MPAA_Rating"].replace({"Open": "Not Rated"})
df["Source"] = df["Source"].replace(dict.fromkeys(rare_source, "Other"))
df["Creative_Type"] = df["Creative_Type"].replace({"Multiple Creative Types": "Other"})
df["Major_Genre"] = df["Major_Genre"].replace({"Concert/Performance": "Other"})

# 5. Casos completos nas categóricas
cat_cols = ["MPAA_Rating", "Major_Genre", "Source", "Creative_Type", "Distributor"]
df = df.dropna(subset=cat_cols)

# 6. Estação de lançamento (regra fixa de domínio)
df["release_season"] = df["release_month"].map(
    lambda m: "summer" if m in (5, 6, 7) else "holiday" if m in (11, 12) else "other")

# 7. Split temporal (80% mais antigo = treino)
model_df = df.sort_values("Release_Date_fixed").reset_index(drop=True)
cut = model_df["Release_Date_fixed"].quantile(0.8)
train = model_df[model_df["Release_Date_fixed"] <= cut].copy()
test = model_df[model_df["Release_Date_fixed"] > cut].copy()

CS = ["MPAA_Rating", "Major_Genre", "Source", "Creative_Type", "release_season"]
CD = ["Distributor"]
CN = ["Production_Budget"]
X_COLS = CS + CD + CN
y_train, y_test = train["success"], test["success"]

print(len(model_df), "filmes |", len(train), "treino |", len(test), "teste | corte:", cut.date())
print("sucesso treino:", round(y_train.mean(), 3), "| sucesso teste:", round(y_test.mean(), 3))

2164 filmes | 1731 treino | 433 teste | corte: 2007-04-22
sucesso treino: 0.657 | sucesso teste: 0.704


# Setup comum

- **Pré-processamento:** one-hot com níveis raros agrupados (menos de 10 linhas no treino; menos de 20 para `Distributor`), dentro de um `Pipeline`, para que o encoder seja ajustado só nos dados de treino de cada fold.
- **Validação:** a mesma *expanding window* do LAB anterior. Para cada ano de 2002 a 2007, treina nos anos anteriores e valida naquele ano. O conjunto de teste não é usado para escolher nada.
- **Referência:** a árvore única escolhida antes (depth 4, leaf 40), com CV AUC de cerca de 0,567.

In [ ]:
def make_prep(scale=False):
    """One-hot com agrupamento de níveis raros (ajustado só no treino de cada fold)."""
    num = StandardScaler() if scale else "passthrough"
    return ColumnTransformer([
        ("cat", OneHotEncoder(min_frequency=10, handle_unknown="infrequent_if_exist", sparse_output=False), CS),
        ("dist", OneHotEncoder(min_frequency=20, handle_unknown="infrequent_if_exist", sparse_output=False), CD),
        ("num", num, CN)])

def pipe(estimator, scale=False):
    return Pipeline([("prep", make_prep(scale)), ("model", estimator)])

def expanding_cv(model_fn, df=train, first_val_year=2002):
    """Treina nos anos anteriores e valida em cada ano de 2002 a 2007. Retorna (média AUC, desvio)."""
    aucs = []
    for yr in range(first_val_year, 2008):
        tr, va = df[df["release_year_fixed"] < yr], df[df["release_year_fixed"] == yr]
        m = model_fn().fit(tr[X_COLS], tr["success"])
        aucs.append(roc_auc_score(va["success"], m.predict_proba(va[X_COLS])[:, 1]))
    return np.mean(aucs), np.std(aucs, ddof=1)

# Referência: a árvore escolhida no LAB anterior (depth 4, leaf 40)
tree_fn = lambda: pipe(DecisionTreeClassifier(max_depth=4, min_samples_leaf=40, random_state=42))
auc, sd = expanding_cv(tree_fn)
print(f"Árvore única (d=4, leaf=40)  CV auc={auc:.3f} (±{sd:.3f})")

Árvore única (d=4, leaf=40)  CV auc=0.567 (±0.019)


# Bagging

O `BaggingClassifier` treina várias árvores em paralelo, cada uma numa amostra *bootstrap* (com reposição) do treino, e faz a média das probabilidades. A média reduz a **variância**, mas não o viés. Por isso o bagging funciona melhor com árvores **profundas** (pouco viés, muita variância), ao contrário da árvore única, que precisava ser rasa para não decorar ruído.

**Grid:** profundidade das árvores base em {4, 8, sem limite} × folha mínima em {5, 20, 40}, com 200 árvores.

In [ ]:
rows = []
for depth, leaf in itertools.product([4, 8, None], [5, 20, 40]):
    fn = lambda: pipe(BaggingClassifier(
        DecisionTreeClassifier(max_depth=depth, min_samples_leaf=leaf, random_state=42),
        n_estimators=200, random_state=42, n_jobs=-1))
    auc, sd = expanding_cv(fn)
    rows.append((depth, leaf, auc, sd))

bag_grid = pd.DataFrame(rows, columns=["depth", "leaf", "auc", "auc_std"]).round(3)
print(bag_grid.sort_values("auc", ascending=False).to_string(index=False))

 depth  leaf   auc  auc_std
   NaN     5 0.602    0.045
   8.0     5 0.599    0.049
   8.0    20 0.579    0.049
   4.0     5 0.578    0.043
   NaN    20 0.577    0.054
   4.0    20 0.570    0.051
   4.0    40 0.564    0.044
   8.0    40 0.563    0.052
   NaN    40 0.561    0.056


**Resultados** (média dos 6 anos de validação)

| Depth | Leaf | AUC | ± entre folds |
|---|---|---|---|
| **sem limite** | **5** | **0,602** | **0,045** |
| 8 | 5 | 0,599 | 0,049 |
| 8 | 20 | 0,579 | 0,049 |
| 4 | 5 | 0,578 | 0,043 |
| ... | ... | ... | ... |
| sem limite | 40 | 0,561 | 0,056 |

- O padrão é o que a teoria prevê: **quanto mais flexíveis as árvores base (folha pequena), melhor o bagging**. Com folha 40, o bagging fica no nível da árvore única (cerca de 0,56).
- O melhor bagging (0,602) supera a árvore única na validação (0,567). A diferença é de cerca de 1,5 erro padrão (0,045/√6 ≈ 0,018), então é sugestiva, mas não conclusiva.

**Decisão:** árvores sem limite de profundidade com folha mínima 5.

In [ ]:
for n in [1, 10, 50, 100, 200, 400]:
    fn = lambda: pipe(BaggingClassifier(
        DecisionTreeClassifier(min_samples_leaf=5, random_state=42),
        n_estimators=n, random_state=42, n_jobs=-1))
    auc, sd = expanding_cv(fn)
    print(f"n_estimators={n:4d}  auc={auc:.3f} (±{sd:.3f})")

n_estimators=   1  auc=0.578 (±0.057)
n_estimators=  10  auc=0.610 (±0.043)
n_estimators=  50  auc=0.605 (±0.050)
n_estimators= 100  auc=0.607 (±0.046)
n_estimators= 200  auc=0.602 (±0.045)
n_estimators= 400  auc=0.600 (±0.046)


**Efeito do número de árvores**

| n_estimators | 1 | 10 | 50 | 100 | 200 | 400 |
|---|---|---|---|---|---|---|
| AUC | 0,578 | 0,610 | 0,605 | 0,607 | 0,602 | 0,600 |
| ± | 0,057 | 0,043 | 0,050 | 0,046 | 0,045 | 0,046 |

- Com **1 árvore** (uma árvore profunda numa amostra bootstrap), o AUC é 0,578 e o desvio é o maior da tabela (0,057). Agregar **10 árvores** já sobe o AUC para cerca de 0,61 e reduz o desvio. Essa é a redução de variância do bagging.
- Depois de cerca de 10 árvores a curva fica plana. As diferenças entre 10 e 400 estão dentro do ruído. Mais árvores não causam overfitting, só custam tempo.

**Decisão:** mantemos 200 árvores, um valor folgado e estável.

# Boosting

O boosting treina árvores **em sequência**: cada nova árvore ajusta os erros (gradientes da log-loss) do conjunto atual. O foco é reduzir o **viés**, por isso as árvores base são rasas. Usamos o `HistGradientBoostingClassifier`, a implementação rápida do gradient boosting no scikit-learn.

Como o sinal é fraco, o boosting pode facilmente ajustar ruído. Por isso usamos uma *learning rate* baixa, árvores rasas, folha mínima alta e regularização L2.

**Grid:** learning rate em {0,03; 0,1} × profundidade em {2, 3} × folha mínima em {20, 40}, com 200 iterações e `l2_regularization=1.0`.

In [ ]:
rows = []
for lr, depth, leaf in itertools.product([0.03, 0.1], [2, 3], [20, 40]):
    fn = lambda: pipe(HistGradientBoostingClassifier(
        learning_rate=lr, max_depth=depth, min_samples_leaf=leaf,
        max_iter=200, l2_regularization=1.0, random_state=42))
    auc, sd = expanding_cv(fn)
    rows.append((lr, depth, leaf, auc, sd))

boost_grid = pd.DataFrame(rows, columns=["lr", "depth", "leaf", "auc", "auc_std"]).round(3)
print(boost_grid.sort_values("auc", ascending=False).to_string(index=False))

  lr  depth  leaf   auc  auc_std
0.10      2    20 0.595    0.046
0.03      3    20 0.594    0.027
0.10      3    20 0.590    0.046
0.10      2    40 0.588    0.049
0.03      2    20 0.587    0.035
0.03      2    40 0.587    0.034
0.10      3    40 0.584    0.052
0.03      3    40 0.582    0.045


**Resultados** (média dos 6 anos de validação)

| LR | Depth | Leaf | AUC | ± entre folds |
|---|---|---|---|---|
| 0,10 | 2 | 20 | 0,595 | 0,046 |
| **0,03** | **3** | **20** | **0,594** | **0,027** |
| 0,10 | 3 | 20 | 0,590 | 0,046 |
| ... | ... | ... | ... | ... |
| 0,03 | 3 | 40 | 0,582 | 0,045 |

- Todas as 8 configurações ficam entre 0,582 e 0,595, acima da árvore única (0,567) e empatadas entre si dentro do ruído.
- **Regra de um erro padrão:** o erro padrão da melhor linha é cerca de 0,046/√6 ≈ 0,019, então todas estão estatisticamente empatadas.

**Decisão:** learning rate 0,03, depth 3, leaf 20. O AUC é praticamente igual ao do topo e o desvio entre folds é o menor da tabela (0,027), o mesmo critério de estabilidade usado na árvore.

# Stacking

O stacking combina modelos **diferentes**. Cada modelo base gera uma probabilidade de sucesso via validação cruzada interna (`cv=5`), e um meta-modelo (regressão logística) aprende a ponderar essas probabilidades.

**Modelos base:** a árvore única (d=4, leaf=40), o bagging e o boosting escolhidos acima, e uma regressão logística (C=0,5). A logística traz um tipo de fronteira diferente das árvores, que é o que dá ao stacking algo para combinar. O orçamento é padronizado (`StandardScaler`) por causa da logística; nas árvores isso não muda nada.

**Observação:** o `cv=5` interno do stacking embaralha os anos dentro do treino. Isso não vaza informação do teste, mas é menos rigoroso que a validação temporal externa, que continua sendo a usada para avaliar o modelo.

In [ ]:
def bagging_est():
    return BaggingClassifier(DecisionTreeClassifier(min_samples_leaf=5, random_state=42),
                             n_estimators=200, oob_score=True, random_state=42, n_jobs=-1)

def boosting_est():
    return HistGradientBoostingClassifier(learning_rate=0.03, max_depth=3, min_samples_leaf=20,
                                          max_iter=200, l2_regularization=1.0, random_state=42)

def stacking_fn():
    base = [("tree", DecisionTreeClassifier(max_depth=4, min_samples_leaf=40, random_state=42)),
            ("bag", bagging_est()),
            ("boost", boosting_est()),
            ("lr", LogisticRegression(C=0.5, max_iter=2000))]
    return pipe(StackingClassifier(estimators=base, final_estimator=LogisticRegression(),
                                   cv=5, stack_method="predict_proba"), scale=True)

auc, sd = expanding_cv(stacking_fn)
print(f"Stacking  CV auc={auc:.3f} (±{sd:.3f})")

Stacking  CV auc=0.591 (±0.041)


**Resumo da validação** (expanding window, treino apenas)

| Modelo | CV AUC | ± entre folds |
|---|---|---|
| Árvore única (d=4, leaf=40) | 0,567 | 0,019 |
| Bagging (árvores profundas, 200) | **0,602** | 0,045 |
| Boosting (lr=0,03, d=3) | 0,594 | 0,027 |
| Stacking | 0,591 | 0,041 |

Na validação, os três ensembles superam a árvore única por cerca de 0,03, e o bagging fica na frente. Com os modelos fixados, avaliamos agora **uma única vez** no conjunto de teste.

# Avaliação final no conjunto de teste

- Cada modelo é treinado no treino inteiro (1.731 filmes) e avaliado nos 433 filmes de teste (abr/2007 a 2010).
- Reportamos o limiar padrão 0,5 e o limiar ajustado do LAB anterior (a taxa de sucesso do treino, 0,657).

In [ ]:
models = {
    "majority dummy": pipe(DummyClassifier(strategy="most_frequent")),
    "árvore única":   tree_fn(),
    "bagging":        pipe(bagging_est()),
    "boosting":       pipe(boosting_est()),
    "stacking":       stacking_fn(),
}

thr = y_train.mean()   # mesmo limiar ajustado do LAB anterior (taxa de sucesso do treino)
probas = {}
for name, m in models.items():
    m.fit(train[X_COLS], y_train)
    p = m.predict_proba(test[X_COLS])[:, 1]
    probas[name] = p
    pred, pred_thr = m.predict(test[X_COLS]), (p >= thr).astype(int)
    print(f"{name:15s} auc={roc_auc_score(y_test, p):.3f} | thr=0.5: acc={accuracy_score(y_test, pred):.3f} "
          f"bal_acc={balanced_accuracy_score(y_test, pred):.3f} | thr={thr:.3f}: acc={accuracy_score(y_test, pred_thr):.3f} "
          f"bal_acc={balanced_accuracy_score(y_test, pred_thr):.3f} f1={f1_score(y_test, pred_thr):.3f}")

print("\nBagging OOB accuracy (treino):", round(models["bagging"]["model"].oob_score_, 3))
print("Pesos do meta-modelo do stacking [tree, bag, boost, lr]:",
      models["stacking"]["model"].final_estimator_.coef_.round(2))
print("\nMatriz de confusão do boosting (thr ajustado):")
print(confusion_matrix(y_test, (probas["boosting"] >= thr).astype(int)))

majority dummy  auc=0.500 | thr=0.5: acc=0.704 bal_acc=0.500 | thr=0.657: acc=0.704 bal_acc=0.500 f1=0.827
árvore única    auc=0.632 | thr=0.5: acc=0.704 bal_acc=0.500 | thr=0.657: acc=0.503 bal_acc=0.566 f1=0.540
bagging         auc=0.577 | thr=0.5: acc=0.610 bal_acc=0.510 | thr=0.657: acc=0.503 bal_acc=0.564 f1=0.542
boosting        auc=0.637 | thr=0.5: acc=0.684 bal_acc=0.531 | thr=0.657: acc=0.545 bal_acc=0.607 f1=0.585
stacking        auc=0.630 | thr=0.5: acc=0.684 bal_acc=0.512 | thr=0.657: acc=0.554 bal_acc=0.611 f1=0.599

Bagging OOB accuracy (treino): 0.649
Pesos do meta-modelo do stacking [tree, bag, boost, lr]: [[0.77 1.23 0.89 0.91]]

Matriz de confusão do boosting (thr ajustado):
[[ 97  31]
 [166 139]]


**Resultados no teste**

| Modelo | AUC | Acc (0,5) | Bal. acc (0,5) | Acc (0,657) | Bal. acc (0,657) | F1 (0,657) |
|---|---|---|---|---|---|---|
| Majority dummy | 0,500 | 0,704 | 0,500 | 0,704 | 0,500 | 0,827 |
| Árvore única | 0,632 | 0,704 | 0,500 | 0,503 | 0,566 | 0,540 |
| Bagging | 0,577 | 0,610 | 0,510 | 0,503 | 0,564 | 0,542 |
| **Boosting** | **0,637** | 0,684 | **0,531** | 0,545 | 0,607 | 0,585 |
| Stacking | 0,630 | 0,684 | 0,512 | 0,554 | **0,611** | **0,599** |

- **O bagging, melhor na validação, foi o pior no teste** (AUC 0,577). A árvore única fez o caminho inverso: 0,567 na validação e 0,632 no teste. Com 433 filmes, o AUC de um único corte varia cerca de ±0,05, então os dois resultados de teste estão provavelmente desviados em direções opostas. Isso mostra por que não se escolhe modelo pelo teste.
- **Boosting e stacking ficam empatados com a árvore em AUC**, mas suas probabilidades são mais bem distribuídas. No limiar 0,5 eles já preveem alguns fracassos (a árvore previa "sucesso" para todos), e no limiar ajustado a balanced accuracy sobe de 0,566 para cerca de 0,61, com mais acurácia (0,55 contra 0,50).
- **Boosting no limiar ajustado:** detecta 97 dos 128 fracassos (recall de 76%) e erra 166 sucessos.
- **OOB do bagging:** a acurácia *out-of-bag* no treino é 0,649, abaixo da taxa de sucesso do treino (0,657). É mais um sinal de que o bagging de árvores profundas mal ganha do chute majoritário em decisões duras.
- **Stacking:** o meta-modelo dá peso positivo a todos os modelos base, com o maior peso no bagging (1,23). Na prática, o stacking fica muito parecido com o boosting.

# Os ensembles ganham da árvore única? (bootstrap pareado)

Mesmo procedimento do LAB anterior: 2.000 reamostragens do teste, calculando em cada uma a diferença de AUC (ensemble - árvore única) nos mesmos filmes.

In [ ]:
rng = np.random.default_rng(42)
yt = y_test.to_numpy()
for name in ["bagging", "boosting", "stacking"]:
    diffs = []
    for _ in range(2000):
        idx = rng.integers(0, len(yt), len(yt))
        diffs.append(roc_auc_score(yt[idx], probas[name][idx]) - roc_auc_score(yt[idx], probas["árvore única"][idx]))
    lo, med, hi = np.percentile(diffs, [2.5, 50, 97.5])
    print(f"{name:9s} - árvore: mediana={med:+.3f}  IC95%=[{lo:+.3f}, {hi:+.3f}]")

bagging   - árvore: mediana=-0.055  IC95%=[-0.113, +0.005]
boosting  - árvore: mediana=+0.006  IC95%=[-0.045, +0.057]
stacking  - árvore: mediana=-0.001  IC95%=[-0.046, +0.046]


**Resultado**

| Comparação | Mediana do ganho em AUC | IC 95% |
|---|---|---|
| Bagging − árvore | −0,055 | [−0,113; +0,005] |
| Boosting − árvore | +0,006 | [−0,045; +0,057] |
| Stacking − árvore | −0,001 | [−0,046; +0,046] |

Todos os intervalos contêm zero, então **nenhum ensemble ranqueia os filmes de teste de forma significativamente diferente da árvore única**. O bagging chega perto de ser significativamente pior.

# Conclusão

**Pergunta.** Ensembles extraem mais sinal pré-lançamento sobre o sucesso de um filme do que uma única árvore de decisão?

**Resposta.** Pouco, e o ganho não é confiável. Na validação temporal, os três ensembles superaram a árvore em cerca de 0,03 de AUC (0,59-0,60 contra 0,567). No teste, nenhum ganho foi significativo. O que os ensembles trazem de concreto é melhor **calibração**: boosting e stacking geram probabilidades mais espalhadas e, no limiar ajustado, chegam a balanced accuracy de cerca de 0,61 (contra 0,566 da árvore), com acurácia maior.

**Sobre cada método**
- **Bagging:** se comportou como a teoria prevê. Funcionou melhor com árvores profundas, ganhou de 1 para 10 árvores (redução de variância) e estabilizou depois disso. Mas reduzir variância não cria sinal: com um sinal fraco e uma mudança de distribuição entre treino (1996-2007) e teste (2007-2010), árvores profundas aprenderam padrões que não se repetiram.
- **Boosting:** o modelo mais consistente. Ficou bem na validação (com o menor desvio entre os ensembles), teve o maior AUC de teste (0,637) e o melhor equilíbrio de recall entre as classes.
- **Stacking:** praticamente igual ao boosting, com a melhor balanced accuracy e o melhor F1 no limiar ajustado, mas com mais complexidade sem ganho claro.

**Escolha final:** se fosse para usar um modelo, seria o **boosting** (lr=0,03, depth 3, leaf 20), que tem desempenho igual ou melhor que o do stacking e é mais simples.

**Limitações:** as mesmas do LAB anterior. Há um único corte temporal com 433 filmes (AUC com cerca de ±0,05 de incerteza), dados só até 2010 em dólares nominais, e o resultado vale apenas para filmes com metadados completos. O sucesso depende muito de fatores fora dos dados (elenco, marketing, críticas, concorrência), o que limita qualquer modelo, ensemble ou não.